# EDA & feature engineering — Seguro de Vida Individual

Partimos de los datos limpios en `data/processed/clean/` — resultado de la extracción
de fuentes CNSF 2021–2024 y la promoción de encabezados reales documentada en
`data_cleaning.ipynb`.

Este notebook cubre tres etapas:

1. **Inspección** — tipos, rangos, nulos y distribuciones por tabla.
2. **Join inferido** — identificar la clave compuesta que permite cruzar `emision`,
   `siniestros` y `comisiones` sin foreign key explícito.
3. **Feature engineering** — construir métricas por segmento (siniestralidad,
   frecuencia, severidad, prima por asegurado) como inputs válidos para modelos de
   pricing.

Ver `docs/ANTECEDENTES.md` para contexto completo del proyecto.

---
## 0. Carga de datos

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN = Path("../data/processed/clean")

emision_f    = pd.read_csv(CLEAN / "emision_clean.csv",    dtype=str)
siniestros_f = pd.read_csv(CLEAN / "siniestros_clean.csv", dtype=str)
comisiones_f = pd.read_csv(CLEAN / "comisiones_clean.csv", dtype=str)

frames = {"emision": emision_f, "siniestros": siniestros_f, "comisiones": comisiones_f}

for name, df in frames.items():
    print(f"✓ {name:<12} {df.shape}")

---
## 1. Inspección — esquema de columnas y tipos raw

Todas las columnas llegaron como `str`. Aquí vemos qué existe en cada tabla
antes de cualquier casteo.

In [ ]:
for name, df in frames.items():
    print(f"\n{'='*60}")
    print(f"  {name.upper()}  — {df.shape[0]:,} filas x {df.shape[1]} columnas")
    print(f"{'='*60}")
    print(df.dtypes.to_string())

---
## 2. Casteo de tipos

Separamos las columnas en dos grupos por tabla:
- **Categóricas / clave**: `ANIO`, `ENTIDAD`, `SEXO`, `EDAD`, `COBERTURA`, `CLAVE_*`, etc.
- **Numéricas**: primas, sumas aseguradas, siniestros, comisiones, número de asegurados.

Usamos `errors='coerce'` para convertir cadenas inválidas en `NaN` y detectarlas.

In [ ]:
# Reconoziced true headres
for name, df in frames.items():
    print(f"\n[{name}]")
    print(list(df.columns))

In [ ]:
# ── 1. Renombrar y limpiar nombres de columna ───────────────────────────────
def normalize_columns(df):
    """Strip espacios, reemplaza espacios internos por _, uppercase."""
    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
        .str.replace(' ', '_', regex=False)
    )
    # La primera columna es ANIO (quedó como '2021' por el header del Excel)
    first_col = df.columns[0]
    if first_col != 'ANIO':
        df = df.rename(columns={first_col: 'ANIO'})
    return df

for name in frames:
    frames[name] = normalize_columns(frames[name])

emision_f    = frames["emision"]
siniestros_f = frames["siniestros"]
comisiones_f = frames["comisiones"]

# Verificar
for name, df in frames.items():
    print(f"\n[{name}]")
    print(list(df.columns))

In [ ]:
# ── Columnas numéricas conocidas por tabla ──────────────────────────────────
NUM_COLS = {
    "emision": [
        "PRIMA_EMITIDA",
        "SUMA_ASEGURADA",
        "NUMERO_DE_ASEGURADOS",
    ],
    "siniestros": [
        "NUMERO_DE_SINIESTROS",
        "MONTO_RECLAMADO",
        "MONTO_PAGADO",
        "VENCIMIENTOS",
        "MONTO_DE_REASEGURO",
    ],
    "comisiones": [
        "NUMERO_DE_ASEGURADOS",
        "PRIMA_CEDIDA",
        "COMISIONES_DIRECTAS",
        "FONDO_DE_INVERSIÓN",
        "FONDO_DE_ADMINISTRACION",
        "MONTO_DE_DIVIDENDOS",
        "MONTO_DE_RESCATE",
    ],
}

def cast_numerics(df, num_cols):
    """Castea a float las columnas numéricas que existan en df."""
    existing = [c for c in num_cols if c in df.columns]
    missing  = [c for c in num_cols if c not in df.columns]
    if missing:
        print(f"  ⚠️  Columnas no encontradas: {missing}")
    for c in existing:
        df[c] = pd.to_numeric(df[c].str.strip().str.replace(',', '', regex=False),
                              errors='coerce')
    return df

for name, df in frames.items():
    print(f"\n[{name}]")
    frames[name] = cast_numerics(df, NUM_COLS.get(name, []))

# Reasignar variables individuales
emision_f    = frames["emision"]
siniestros_f = frames["siniestros"]
comisiones_f = frames["comisiones"]

print("\n✓ Casteo completado")

In [ ]:
# ── Persistir datasets normalizados ─────────────────────────────────────────
CLEAN_V2 = Path("../data/processed/clean_v2")
CLEAN_V2.mkdir(parents=True, exist_ok=True)

for name, df in frames.items():
    path = CLEAN_V2 / f"{name}_v2.csv"
    df.to_csv(path, index=False)
    print(f"✓ {name:<12} → {path}  {df.shape}")

---
## ⚡ Quick start — sesiones posteriores
Cargar desde `clean_v2/` si el pipeline de normalización ya fue ejecutado.
Saltar las secciones 0–2 si se parte desde aquí.

In [9]:
import pandas as pd
from pathlib import Path

CLEAN_V2 = Path("../data/processed/clean_v2")

emision_f    = pd.read_csv(CLEAN_V2 / "emision_v2.csv")
siniestros_f = pd.read_csv(CLEAN_V2 / "siniestros_v2.csv")
comisiones_f = pd.read_csv(CLEAN_V2 / "comisiones_v2.csv")

frames = {"emision": emision_f, "siniestros": siniestros_f, "comisiones": comisiones_f}

for name, df in frames.items():
    print(f"✓ {name:<12} {df.shape}")

/tmp/ipykernel_398074/2223794813.py:6: DtypeWarning: Columns (0: EDAD) have mixed types. Specify dtype option on import or set low_memory=False.
  emision_f    = pd.read_csv(CLEAN_V2 / "emision_v2.csv")
/tmp/ipykernel_398074/2223794813.py:8: DtypeWarning: Columns (0: EDAD) have mixed types. Specify dtype option on import or set low_memory=False.
  comisiones_f = pd.read_csv(CLEAN_V2 / "comisiones_v2.csv")


✓ emision      (3508592, 12)
✓ siniestros   (297504, 13)
✓ comisiones   (827906, 16)


In [11]:
print(f"The types of emision_f is: {emision_f.dtypes}")

The types of emision_f is: ANIO                        int64
EDAD                       object
COBERTURA                     str
PLAN_DE_LA_POLIZA             str
MODALIDAD_DE_LA_POLIZA        str
MONEDA                        str
ENTIDAD                       str
SEXO                          str
FORMA_DE_VENTA                str
NUMERO_DE_ASEGURADOS      float64
PRIMA_EMITIDA             float64
SUMA_ASEGURADA            float64
dtype: object


In [15]:
print(emision_f["ANIO"].describe())
print("==========================================\n" \
"Estadisticas descriptivas \ng" \
"==========================================")
print(emision_f.ANIO.value_counts().sort_index())
print(emision_f.ANIO.min())
print(emision_f.ANIO.max())

count    3.508592e+06
mean     2.022547e+03
std      1.113580e+00
min      2.021000e+03
25%      2.022000e+03
50%      2.023000e+03
75%      2.024000e+03
max      2.024000e+03
Name: ANIO, dtype: float64
Estadisticas descriptivas 
g==========================================
ANIO
2021    816498
2022    879534
2023    888269
2024    924291
Name: count, dtype: int64
2021
2024


In [10]:
print(f"The dimension of emision_f is: {emision_f.shape}")
print(f"The types of emision_f is: {emision_f.dtypes}")
print(f"Describe emision_f: {emision_f.describe()}")
print(f"Sum is null in emision_f: {emision_f.isnull().sum()}")

The dimension of emision_f is: (3508592, 12)
The types of emision_f is: ANIO                        int64
EDAD                       object
COBERTURA                     str
PLAN_DE_LA_POLIZA             str
MODALIDAD_DE_LA_POLIZA        str
MONEDA                        str
ENTIDAD                       str
SEXO                          str
FORMA_DE_VENTA                str
NUMERO_DE_ASEGURADOS      float64
PRIMA_EMITIDA             float64
SUMA_ASEGURADA            float64
dtype: object
Describe emision_f:                ANIO  NUMERO_DE_ASEGURADOS  PRIMA_EMITIDA  SUMA_ASEGURADA
count  3.508592e+06          3.508589e+06   3.508589e+06    3.508589e+06
mean   2.022547e+03          6.000479e+01   2.480917e+05    1.945208e+07
std    1.113580e+00          5.181768e+02   3.198568e+06    1.026886e+08
min    2.021000e+03          1.000000e+00  -1.489461e+07    0.000000e+00
25%    2.022000e+03          2.000000e+00   4.483200e+02    6.000843e+04
50%    2.023000e+03          5.000000e+00   4.39

---
## 3. Estadísticas descriptivas — columnas numéricas

Para cada tabla: `count`, `mean`, `std`, `min`, `p25`, `p50`, `p75`, `p95`, `max`.

El percentil 95 es clave para detectar colas largas típicas en seguros.

In [3]:
PERCENTILES = [0.25, 0.50, 0.75, 0.95]

for name, df in frames.items():
    num_df = df.select_dtypes(include='number')
    if num_df.empty:
        print(f"\n[{name}] — sin columnas numéricas casteadas")
        continue
    desc = num_df.describe(percentiles=PERCENTILES).T
    desc.columns = [c.upper() for c in desc.columns]   # estandarizar nombres
    print(f"\n{'='*70}")
    print(f"  {name.upper()} — estadísticas numéricas")
    print(f"{'='*70}")
    print(desc.to_string(float_format=lambda x: f"{x:,.2f}"))


  EMISION — estadísticas numéricas
                            COUNT          MEAN            STD            MIN       25%        50%          75%           95%               MAX
ANIO                 3,508,592.00      2,022.55           1.11       2,021.00  2,022.00   2,023.00     2,024.00      2,024.00          2,024.00
NUMERO_DE_ASEGURADOS 3,508,589.00         60.00         518.18           1.00      2.00       5.00        23.00        257.00        130,071.00
PRIMA_EMITIDA        3,508,589.00    248,091.69   3,198,567.96 -14,894,612.05    448.32   4,394.01    37,568.77    635,092.05  2,448,270,922.42
SUMA_ASEGURADA       3,508,589.00 19,452,081.65 102,688,613.09           0.00 60,008.43 995,821.00 6,469,399.00 81,257,198.08 66,326,216,079.25

  SINIESTROS — estadísticas numéricas
                          COUNT       MEAN          STD             MIN      25%       50%        75%          95%            MAX
ANIO                 297,504.00   2,022.45         1.13        2,021.00 2,0

---
## 4. Diagnóstico de nulos

Reportamos el porcentaje de nulos por columna en cada tabla. Un nulo en una
columna numérica post-casteo puede indicar:
- valor original vacío (`''`), o
- cadena no parseable (e.g. `'N/D'`, `'-'`, valores con formato incorrecto).

Ambos casos requieren decisión antes del join o feature engineering.

In [4]:
def null_report(df, name):
    total = len(df)
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / total * 100).round(2)
    report = pd.DataFrame({
        "nulos": null_counts,
        "pct_%": null_pct,
        "dtype": df.dtypes,
    })
    report = report[report["nulos"] > 0].sort_values("pct_%", ascending=False)
    print(f"\n{'='*55}")
    print(f"  {name.upper()} — columnas con nulos ({total:,} filas total)")
    print(f"{'='*55}")
    if report.empty:
        print("  ✅ Sin nulos detectados")
    else:
        print(report.to_string())

for name, df in frames.items():
    null_report(df, name)


  EMISION — columnas con nulos (3,508,592 filas total)
                      nulos  pct_%    dtype
EDAD                     11    0.0   object
NUMERO_DE_ASEGURADOS      3    0.0  float64
PRIMA_EMITIDA             3    0.0  float64
SUMA_ASEGURADA            3    0.0  float64

  SINIESTROS — columnas con nulos (297,504 filas total)
                      nulos  pct_%    dtype
PLAN_DE_LA_POLIZA         1    0.0      str
NUMERO_DE_SINIESTROS      3    0.0  float64
MONTO_RECLAMADO           3    0.0  float64
VENCIMIENTOS              3    0.0  float64
MONTO_PAGADO              3    0.0  float64
MONTO_DE_REASEGURO        3    0.0  float64

  COMISIONES — columnas con nulos (827,906 filas total)
                         nulos  pct_%    dtype
NUMERO_DE_ASEGURADOS         3    0.0  float64
PRIMA_CEDIDA                 3    0.0  float64
COMISIONES_DIRECTAS          3    0.0  float64
FONDO_DE_INVERSIÓN           3    0.0  float64
FONDO_DE_ADMINISTRACION      3    0.0  float64
MONTO_DE_DIVIDENDOS 

---
## 5. Cardinalidad y valores únicos — columnas categóricas

Para las columnas tipo `object` (posibles claves de segmentación), reportamos:
- cardinalidad (# valores únicos),
- los top-10 valores por frecuencia.

Esto permite identificar qué columnas son candidatas a clave compuesta de join.

In [5]:
def categorical_report(df, name, top_n=10):
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    print(f"\n{'='*60}")
    print(f"  {name.upper()} — columnas categóricas ({len(cat_cols)} cols)")
    print(f"{'='*60}")
    for col in cat_cols:
        vc = df[col].value_counts(dropna=False)
        card = df[col].nunique(dropna=False)
        null_n = df[col].isnull().sum()
        print(f"\n  ▸ {col}  (cardinalidad={card:,}, nulos={null_n:,})")
        print(vc.head(top_n).to_string())

for name, df in frames.items():
    categorical_report(df, name)

/tmp/ipykernel_398074/3816428377.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()



  EMISION — columnas categóricas (8 cols)

  ▸ EDAD  (cardinalidad=248, nulos=11)
EDAD
49    41511
48    41225
47    41207
45    41147
44    40943
50    40901
46    40856
43    40687
42    40654
40    40543

  ▸ COBERTURA  (cardinalidad=21, nulos=0)
COBERTURA
Fallecimiento                              884590
Invalidez total y permanente               398082
Muerte accidental (Doble indemnización)    395521
Exención de pago de prima                  324284
Enfermedades graves                        230371
Ahorro / inversión                         164187
Muerte colectiva (Triple indemnización)    160176
Gastos funerarios                          159373
Sobrevivencia                              150254
Otros                                      128565

  ▸ PLAN_DE_LA_POLIZA  (cardinalidad=10, nulos=0)
PLAN_DE_LA_POLIZA
Temporal                   1749273
Vitalicio                   972365
Dotal Mixto                 755649
Rentas                       19292
Otros                         

/tmp/ipykernel_398074/3816428377.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()
/tmp/ipykernel_398074/3816428377.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select


  ▸ MODALIDAD_DE_LA_POLIZA  (cardinalidad=11, nulos=0)
MODALIDAD_DE_LA_POLIZA
Tradicional                      431124
Flexible sin tasa garantizada    152801
Flexible con tasa garantizada    118768
Educativo                         60932
Deudores                          37863
Producto básico estandarizado      9841
Mancomunado                        7661
Retiro                             6531
Microseguro                        2208
Otros                               174

  ▸ MONEDA  (cardinalidad=4, nulos=0)
MONEDA
Nacional      454156
Indizada      188525
Extranjera    185222
MONEDA             3

  ▸ ENTIDAD  (cardinalidad=36, nulos=0)
ENTIDAD
Ciudad de México    62123
Mexico              46592
Jalisco             41493
Nuevo Leon          37202
Guanajuato          32109
Puebla              31936
Veracruz            31436
Querétaro           28008
Chihuahua           27668
Baja California     26853

  ▸ SEXO  (cardinalidad=3, nulos=0)
SEXO
Masculino    441494
Femenino     386409


---
## 6. Cobertura temporal — distribución por ANIO

Verificamos que los cuatro años (2021–2024) estén representados de forma
homogénea en las tres tablas. Asimetrías pueden indicar datos faltantes
o diferencias en el reporte por año.

In [6]:
anio_col = "ANIO"   # ajustar si el nombre difiere

print(f"{'TABLA':<15} {'ANIO':<8} {'FILAS':>10} {'PCT':>8}")
print("-" * 45)

for name, df in frames.items():
    if anio_col not in df.columns:
        print(f"  ⚠️  {name}: columna '{anio_col}' no encontrada")
        continue
    vc = df[anio_col].value_counts().sort_index()
    total = len(df)
    for anio, cnt in vc.items():
        pct = cnt / total * 100
        print(f"{name:<15} {str(anio):<8} {cnt:>10,} {pct:>7.1f}%")
    print("-" * 45)

TABLA           ANIO          FILAS      PCT
---------------------------------------------
emision         2021        816,498    23.3%
emision         2022        879,534    25.1%
emision         2023        888,269    25.3%
emision         2024        924,291    26.3%
---------------------------------------------
siniestros      2021         81,107    27.3%
siniestros      2022         72,623    24.4%
siniestros      2023         71,236    23.9%
siniestros      2024         72,538    24.4%
---------------------------------------------
comisiones      2021        211,493    25.5%
comisiones      2022        204,955    24.8%
comisiones      2023        200,255    24.2%
comisiones      2024        211,203    25.5%
---------------------------------------------


---
## 7. Columnas compartidas — candidatas a clave compuesta

Identificamos las columnas presentes en las tres tablas simultáneamente.
Son las candidatas naturales para el join inferido del siguiente bloque.

In [7]:
col_sets = {name: set(df.columns) for name, df in frames.items()}

shared_all   = col_sets["emision"] & col_sets["siniestros"] & col_sets["comisiones"]
shared_em_si = col_sets["emision"] & col_sets["siniestros"]
shared_em_co = col_sets["emision"] & col_sets["comisiones"]
shared_si_co = col_sets["siniestros"] & col_sets["comisiones"]

print("Columnas en las TRES tablas:")
print(sorted(shared_all))

print("\nColumnas sólo en emision ∩ siniestros (no en comisiones):")
print(sorted(shared_em_si - col_sets["comisiones"]))

print("\nColumnas sólo en emision ∩ comisiones (no en siniestros):")
print(sorted(shared_em_co - col_sets["siniestros"]))

print("\nColumnas exclusivas por tabla:")
for name, cs in col_sets.items():
    excl = cs - (shared_em_si | shared_em_co | shared_si_co)
    print(f"  {name}: {sorted(excl)}")

Columnas en las TRES tablas:
['ANIO', 'EDAD', 'ENTIDAD', 'MODALIDAD_DE_LA_POLIZA', 'PLAN_DE_LA_POLIZA', 'SEXO']

Columnas sólo en emision ∩ siniestros (no en comisiones):
['COBERTURA']

Columnas sólo en emision ∩ comisiones (no en siniestros):
['FORMA_DE_VENTA', 'MONEDA', 'NUMERO_DE_ASEGURADOS']

Columnas exclusivas por tabla:
  emision: ['PRIMA_EMITIDA', 'SUMA_ASEGURADA']
  siniestros: ['CAUSA_DEL_SINIESTRO', 'MONTO_DE_REASEGURO', 'MONTO_PAGADO', 'MONTO_RECLAMADO', 'NUMERO_DE_SINIESTROS', 'VENCIMIENTOS']
  comisiones: ['COMISIONES_DIRECTAS', 'FONDO_DE_ADMINISTRACION', 'FONDO_DE_INVERSIÓN', 'MONTO_DE_DIVIDENDOS', 'MONTO_DE_RESCATE', 'PRIMA_CEDIDA', 'TIPO_DIVIDENDO']


---
## 8. Resumen ejecutivo — criterio de homogeneidad

Con base en los resultados anteriores evaluamos si las tres tablas corresponden
a la **misma cartera** y si la granularidad permite construir features válidos.

Criterios de homogeneidad:

| Criterio | Indicador |
|---|---|
| Cobertura temporal | Mismos años en las tres tablas |
| Entidades reportantes | Mismo conjunto de `ENTIDAD` |
| Segmentación demográfica | `SEXO`, `EDAD`, `COBERTURA` consistentes |
| Escala numérica | Rangos razonables para seguros de vida individual MXN |
| Nulos | Ausencia o patrón sistemático explicable |


In [8]:
# ── Comparación de entidades entre tablas ───────────────────────────────────
entidad_col = "ENTIDAD"   # ajustar si el nombre difiere

entidades = {}
for name, df in frames.items():
    if entidad_col in df.columns:
        entidades[name] = set(df[entidad_col].dropna().unique())

if len(entidades) == 3:
    names = list(entidades.keys())
    common = entidades[names[0]] & entidades[names[1]] & entidades[names[2]]
    print(f"Entidades comunes a las 3 tablas: {len(common)}")
    for name, ents in entidades.items():
        solo = ents - common
        print(f"  {name}: {len(ents)} entidades totales, {len(solo)} exclusivas → {sorted(solo) if solo else '—'}")
else:
    print("⚠️  Columna ENTIDAD no encontrada en alguna tabla — revisar nombre")

# ── Resumen de cardinalidad de claves demográficas ──────────────────────────
print()
demo_cols = ["SEXO", "EDAD", "COBERTURA"]
print(f"{'COLUMNA':<15}", end="")
for name in frames:
    print(f"  {name:>12}", end="")
print()
print("-" * (15 + 15 * len(frames)))

for col in demo_cols:
    print(f"{col:<15}", end="")
    for name, df in frames.items():
        if col in df.columns:
            card = df[col].nunique()
            print(f"  {card:>12,}", end="")
        else:
            print(f"  {'—':>12}", end="")
    print()

Entidades comunes a las 3 tablas: 35
  emision: 38 entidades totales, 3 exclusivas → ['ENTIDAD ', 'No disponible', 'No disponible ']
  siniestros: 36 entidades totales, 1 exclusivas → ['ENTIDAD']
  comisiones: 36 entidades totales, 1 exclusivas → ['ENTIDAD ']

COLUMNA               emision    siniestros    comisiones
------------------------------------------------------------
SEXO                        4             5             3
EDAD                      247           117           198
COBERTURA                  21            21             —
